In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split, TensorDataset
from torchvision import datasets, transforms
import os
import random
import numpy as np

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'


################################
#     RESTART     RUNTIME      #
################################
from ssonn.model.utils import *
from ssonn.metrics.nonlinearity_metrics import *
from ssonn.metrics.edge_finder import *
from ssonn.metrics.train_metrics import *
from ssonn.train.train import *

In [2]:
class SimpleFCN(nn.Module):
    def __init__(self, input_size=28 * 28, hidden_size=16):
        super(SimpleFCN, self).__init__()
        self.fc0 = nn.Linear(input_size, 10)
        # self.fc1 = nn.Linear(hidden_size, 10)
        # self.act = nn.ReLU()

    def forward(self, x):
        x = self.fc0(x)
        return x

In [3]:
hyperparams = {
    "num_epochs": 32,
    "batch_size": 256,
    "edge_importance_metric": AbsGradientEdgeMetric(nn.CrossEntropyLoss()),
    "edge_score_aggregation": "mean",
    "expansion_thresholds": {"fc0": 0.5},
    "pruning_thresholds": {"fc0": 1},
    "plateau_threshold": 0.05,
    "min_epochs_between_expansions": 8,
    # "min_epochs_between_expansions": 10_000,
    "plateau_window_size": 5,
    "learning_rate": 1e-3,
    "prune_after_epochs": 4,
    "task_type": "classification",
    "start_fully_connected": False,
    "max_new_edges_per_expansion": 500,
    "weight_decay": 1e-4
}

In [4]:
# Dataset and Dataloader
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))
])

# Load dataset and split into train/validation sets
dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

train_dataset, val_dataset = random_split(dataset, [0.8, 0.2])

train_loader = DataLoader(train_dataset, batch_size=hyperparams['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=hyperparams['batch_size'], shuffle=False)

In [5]:
model = SimpleFCN()
sparse_model = convert_dense_to_sparse_network(model, layers=[model.fc0], device=device)

In [6]:
# import wandb
# wandb.login()

In [7]:
# wandb.finish()
# run = wandb.init(
#     project="self-expanding-nets",
#     name=f"name",
#     config=hyperparams
# )

In [8]:
from comet_ml import start
from comet_ml.integration.pytorch import log_model
experiment = start(
  api_key="frGbe20bUNVBiHdnVG40YiMwA",
  project_name="general",
  workspace="vanya9090"
)
experiment.log_parameters(hyperparams)

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch, sklearn.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/vanya9090/general/d205848c68394d32a7cbc5f424dd4ac1



In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(sparse_model.parameters(), lr=hyperparams['learning_rate'], weight_decay=hyperparams['weight_decay'])
train_sparse_recursive(sparse_model, train_loader, train_loader, val_loader, criterion,optimizer, hyperparams, experiment, device)

  0%|          | 0/188 [00:00<?, ?it/s]

100%|██████████| 188/188 [00:01<00:00, 100.76it/s]
COMET INFO: Trying to log a figure that is exceeding allowed SVG file size limit (1262372 > 1048576). Falling back to PNG format.


Epoch 1/32, Train Loss: 0.9315, Val Loss: 0.5400, Val Accuracy: 0.8698


100%|██████████| 188/188 [00:01<00:00, 99.20it/s] 
COMET INFO: Trying to log a figure that is exceeding allowed SVG file size limit (1270272 > 1048576). Falling back to PNG format.


Epoch 2/32, Train Loss: 0.4607, Val Loss: 0.4130, Val Accuracy: 0.8934


100%|██████████| 188/188 [00:01<00:00, 97.02it/s] 
COMET INFO: Trying to log a figure that is exceeding allowed SVG file size limit (1272963 > 1048576). Falling back to PNG format.


... (55 outputs left out)

100%|██████████| 188/188 [00:07<00:00, 23.73it/s]
COMET INFO: Trying to log a figure that is exceeding allowed SVG file size limit (2408917 > 1048576). Falling back to PNG format.
COMET INFO: Trying to log a figure that is exceeding allowed SVG file size limit (13628735 > 1048576). Falling back to PNG format.
COMET INFO: Trying to log a figure that is exceeding allowed SVG file size limit (1763109 > 1048576). Falling back to PNG format.
COMET INFO: Trying to log a figure that is exceeding allowed SVG file size limit (1331132 > 1048576). Falling back to PNG format.


Epoch 31/32, Train Loss: 0.0238, Val Loss: 0.0797, Val Accuracy: 0.9783


100%|██████████| 188/188 [00:07<00:00, 24.59it/s]
COMET INFO: Trying to log a figure that is exceeding allowed SVG file size limit (2407414 > 1048576). Falling back to PNG format.
COMET INFO: Trying to log a figure that is exceeding allowed SVG file size limit (13629656 > 1048576). Falling back to PNG format.
COMET INFO: Trying to log a figure that is exceeding allowed SVG file size limit (1765609 > 1048576). Falling back to PNG format.
COMET INFO: Trying to log a figure that is exceeding allowed SVG file size limit (1333199 > 1048576). Falling back to PNG format.


Epoch 32/32, Train Loss: 0.0217, Val Loss: 0.0780, Val Accuracy: 0.9782
torch.Size([7854]) torch.Size([14653])
combined_metrics torch.Size([22507])
mask torch.Size([22507])
tensor(22507)
num_emb_edges 7854
tensor(0) tensor(0)
Chosen edges to del emb: tensor([], size=(2, 0), dtype=torch.int32) 0
Chosen edges to del exp: tensor([], size=(2, 0), dtype=torch.int64) 0


In [10]:
experiment.end()

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : associated_crayfish_9107
COMET INFO:     url                   : https://www.comet.com/vanya9090/general/d205848c68394d32a7cbc5f424dd4ac1
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     acc amount [32]               : (7.95341107221447e-06, 0.00011698554421768707)
COMET INFO:     len_choose [3]                : (102, 500)
COMET INFO:     lr                            : 0.001
COMET INFO:     n_params over train_time [32] : (3896.0942616769344, 54412.11186007656)
COMET INFO:     params amount [32]            : (7840, 122547)
COMET INFO:     params ratio [32]             : (0.9362244897959183, 0.9998036135113905)
COMET INFO:     params to repl

In [11]:
torch.save(sparse_model, 'mnist.pt')